# Tool 2: Wall Mid Calculator & Fair Price Analyzer

## Purpose
Computes and compares different fair-price estimates for each product:
- **Wall Mid**: Average of the deepest-liquidity bid and ask levels (robust)
- **Raw Mid**: Simple (best_bid + best_ask) / 2 (noisy)
- **CSV Mid**: The mid_price from the competition data

## Why This Matters
The Frankfurt Hedgehogs (2nd place, Prosperity 3) called the Wall Mid
"crucial for designing effective strategies." The raw mid-price can be
heavily distorted when bots overbid or undercut aggressively. The Wall Mid
is far more stable because it uses the **deepest liquidity levels**, which
are typically placed by designated market makers who know the true price.

## What You'll See
1. **Time series comparison**: Wall Mid vs Raw Mid vs CSV Mid overlaid
2. **Divergence plot**: When and how much the raw mid deviates from Wall Mid
3. **Stability statistics**: Std dev, range, drift for each estimator
4. **Wall identification**: Which price levels consistently carry the deepest volume
5. **Cross-day comparison**: How each estimator behaves across all 3 days

## How to Use
1. Set `PRODUCT` and `DAY` in the Configuration cell
2. Run all cells
3. Compare the stability of Wall Mid vs Raw Mid — the more stable one is
   the better fair-price estimate for your trading algorithm

## Dependencies
- `data_loader.py` (in same directory)
- `matplotlib`, `pandas`, `numpy`

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

PRODUCT = "ASH_COATED_OSMIUM"  # or "INTARIAN_PEPPER_ROOT"
DAY = -1                        # -2, -1, or 0

# Time range (set TIME_END = None for full day)
TIME_START = 0
TIME_END = None

FIG_WIDTH = 18
FIG_HEIGHT = 5

In [ ]:
# ============================================================
# SETUP
# ============================================================
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import (
    load_prices, compute_wall_mid, compute_raw_mid, filter_time_range,
    AVAILABLE_DAYS, PRODUCTS,
)

prices = load_prices(day=DAY, product=PRODUCT)
prices = compute_wall_mid(prices)
prices = compute_raw_mid(prices)

if TIME_END is not None:
    prices = filter_time_range(prices, TIME_START, TIME_END)

print(f"Loaded {len(prices)} snapshots for {PRODUCT} on Day {DAY}")

In [ ]:
# ============================================================
# PLOT 1: Fair Price Estimators Over Time
# ============================================================
# Compares all three fair-price estimates on one plot.
# Look for: which line is smoothest? Which one jumps around?

fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

ts = prices["timestamp"]
ax.plot(ts, prices["wall_mid"], label="Wall Mid", color="orange", linewidth=1.5)
ax.plot(ts, prices["raw_mid"], label="Raw Mid", color="gray", linewidth=1, alpha=0.7)
ax.plot(ts, prices["mid_price"], label="CSV Mid", color="purple", linewidth=1, alpha=0.5, linestyle=":")

ax.set_xlabel("Timestamp")
ax.set_ylabel("Price")
ax.set_title(f"Fair Price Estimators — {PRODUCT} (Day {DAY})", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PLOT 2: Divergence Between Estimators
# ============================================================
# Shows (Raw Mid - Wall Mid) over time.
# Large spikes = moments where the raw mid is distorted by aggressive
# bots. Your algo should use Wall Mid during these moments.

divergence = prices["raw_mid"] - prices["wall_mid"]

fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))
ax.plot(ts, divergence, color="red", linewidth=0.8, alpha=0.7)
ax.axhline(0, color="black", linewidth=0.5)
ax.fill_between(ts, divergence, 0, alpha=0.2, color="red")

ax.set_xlabel("Timestamp")
ax.set_ylabel("Raw Mid − Wall Mid")
ax.set_title(f"Price Estimator Divergence — {PRODUCT} (Day {DAY})", fontweight="bold")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Divergence stats:")
print(f"  Mean:   {divergence.mean():.3f}")
print(f"  Std:    {divergence.std():.3f}")
print(f"  Max:    {divergence.max():.3f}")
print(f"  Min:    {divergence.min():.3f}")
print(f"  % timesteps where |divergence| > 1: {(divergence.abs() > 1).mean()*100:.1f}%")

In [ ]:
# ============================================================
# PLOT 3: Wall Price Level Identification
# ============================================================
# Shows which price levels carry the deepest liquidity.
# Consistent wall prices = the market maker is anchoring around a known
# true value. The gap between bid wall and ask wall is the "true spread."

fig, axes = plt.subplots(1, 2, figsize=(FIG_WIDTH, FIG_HEIGHT))

# Bid wall prices over time
mask = prices["bid_wall_price"].notna()
axes[0].scatter(prices.loc[mask, "timestamp"], prices.loc[mask, "bid_wall_price"],
                s=prices.loc[mask, "bid_wall_volume"] * 2, c="tab:blue", alpha=0.3)
axes[0].set_title(f"Bid Wall Prices (sized by volume)", fontweight="bold")
axes[0].set_xlabel("Timestamp")
axes[0].set_ylabel("Price")
axes[0].grid(True, alpha=0.3)

# Ask wall prices over time
mask = prices["ask_wall_price"].notna()
axes[1].scatter(prices.loc[mask, "timestamp"], prices.loc[mask, "ask_wall_price"],
                s=prices.loc[mask, "ask_wall_volume"] * 2, c="tab:red", alpha=0.3)
axes[1].set_title(f"Ask Wall Prices (sized by volume)", fontweight="bold")
axes[1].set_xlabel("Timestamp")
axes[1].set_ylabel("Price")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nWall price distribution:")
print("  Bid wall prices:", sorted(prices["bid_wall_price"].dropna().unique()))
print("  Ask wall prices:", sorted(prices["ask_wall_price"].dropna().unique()))

In [ ]:
# ============================================================
# TABLE: Stability Statistics Across All Days
# ============================================================
# Compare how stable each fair-price estimator is across all 3 days.
# Lower std = more stable = better for use as the "true price" in your algo.

rows = []
for d in AVAILABLE_DAYS:
    p = compute_wall_mid(compute_raw_mid(load_prices(day=d, product=PRODUCT)))
    rows.append({
        "Day": d,
        "Wall Mid Mean": p["wall_mid"].mean(),
        "Wall Mid Std": p["wall_mid"].std(),
        "Raw Mid Mean": p["raw_mid"].mean(),
        "Raw Mid Std": p["raw_mid"].std(),
        "CSV Mid Mean": p["mid_price"].mean(),
        "CSV Mid Std": p["mid_price"].std(),
    })

stats_df = pd.DataFrame(rows)
print(f"Stability comparison for {PRODUCT}:\n")
print(stats_df.to_string(index=False, float_format="{:.2f}".format))